# RAG Academic Paper QA System — Colab Demo

This notebook demonstrates the full RAG pipeline using **pre-built indexes and corpus** included in the repository, so all experiment results are reproducible and consistent with the project report.

1. **Environment Setup** — clone repo, install dependencies
2. **Configure API Key** — set Groq API key for LLM generation
3. **Verify Pre-built Data** — check that papers, chunks, and indexes are ready
4. **Single Query Demo** — retrieve → rerank → generate
5. **Retrieval Mode Comparison** — BM25 vs Dense vs Hybrid on the same question
6. **Run Experiments** — automated experiments with chart generation
7. **Display Results** — show all experiment charts

> **Note:** The LLM generation step requires a free Groq API key (get one at https://console.groq.com). Without it, retrieval and reranking still work but answers will show a placeholder.

## 1. Environment Setup

Clone the repository and install all dependencies. The repository already includes the paper corpus (`data/papers/`) and pre-built indexes (`indexes/`), so no additional downloads are needed.

In [ ]:
# Clone the repository (includes papers + pre-built indexes)
!git clone -b init-project https://github.com/xrty/CptS440-540Proj.git
%cd CptS440-540Proj

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Colab's pre-installed packages require numpy>=2.0, override our local pin
!pip install -q "numpy>=2.0" "faiss-cpu>=1.8.0" --force-reinstall

**After the install completes, restart the runtime:**  
Go to **Runtime → Restart session**, then **skip the two cells above** and continue from Section 2.

## 2. Configure Groq API Key

The LLM generation step uses Groq's free API (llama-3.3-70b-versatile).  
Get a free key at: https://console.groq.com

**Option A** — Use Colab Secrets (recommended):  
1. Click the key icon in the left sidebar → Add a secret named `GROQ_API_KEY`  
2. Run the cell below  

**Option B** — Paste directly (less secure):  
Uncomment the line below and paste your key.

In [ ]:
import os

# Option A: Read from Colab Secrets
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded from Colab Secrets.")
except Exception:
    pass

# Option B: Set manually (uncomment and paste your key)
# os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

if os.environ.get("GROQ_API_KEY") and os.environ["GROQ_API_KEY"] != "your_groq_api_key_here":
    print("API key is set. LLM generation will be enabled.")
else:
    print("No API key detected. Retrieval and reranking will still work, but LLM answers will show a placeholder.")

## 3. Verify Pre-built Data

The repository includes 20 ArXiv papers and pre-built BM25 + FAISS indexes.  
This ensures experiment results are identical to those in the project report.

In [ ]:
import sys, json
sys.path.insert(0, "src")
from config import PAPERS_DIR, CHUNKS_PATH, BM25_INDEX_PATH, FAISS_INDEX_PATH, QA_TESTSET_PATH

pdf_count = len(list(PAPERS_DIR.glob("*.pdf")))
print(f"PDFs in corpus   : {pdf_count}")
print(f"chunks.json      : {'OK' if CHUNKS_PATH.exists() else 'MISSING'}")
print(f"BM25 index       : {'OK' if BM25_INDEX_PATH.exists() else 'MISSING'}")
print(f"FAISS index      : {'OK' if FAISS_INDEX_PATH.exists() else 'MISSING'}")

if CHUNKS_PATH.exists():
    chunks = json.loads(CHUNKS_PATH.read_text())
    print(f"Total chunks     : {len(chunks)}")

if QA_TESTSET_PATH.exists():
    testset = json.loads(QA_TESTSET_PATH.read_text())
    factual = sum(1 for q in testset if q.get("type") == "factual")
    semantic = sum(1 for q in testset if q.get("type") == "semantic")
    print(f"Test set         : {len(testset)} questions ({factual} factual, {semantic} semantic)")

## 4. Single Query Demo

Run the full RAG pipeline on a single question:  
**Retrieve** (hybrid, top-20) → **Rerank** (top-5) → **Generate** (Groq LLM)

In [ ]:
from pipeline import RAGPipeline

# Initialize the full pipeline (loads retriever, reranker, generator)
pipeline = RAGPipeline(retrieval_mode="hybrid", top_k=20, rerank_top_k=5)
print("Pipeline loaded successfully.")

In [ ]:
# Ask a question
question = "What is the key contribution of the Transformer model?"
result = pipeline.query(question)

print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nSources ({len(result['sources'])} passages):")
for i, src in enumerate(result["sources"], 1):
    score = src.get("rerank_score", src.get("score", 0))
    print(f"  {i}. {src.get('source_paper', 'Unknown')}, p.{src.get('page_num', '?')} [score={score:.3f}]")

lat = result["latency"]
print(f"\nLatency: retrieval={lat['retrieval_s']}s | rerank={lat['rerank_s']}s | generation={lat['generation_s']}s | total={lat['total_s']}s")

## 5. Retrieval Mode Comparison

Compare BM25, Dense, and Hybrid retrieval on the same question.  
Notice how BM25 matches keywords while Dense captures semantic meaning.

In [ ]:
query = "How does masked language modeling work in pre-training?"

for mode in ["bm25", "dense", "hybrid"]:
    print(f"\n{'='*60}")
    print(f"  Retrieval Mode: {mode.upper()} — Top 3 results")
    print(f"{'='*60}")
    results = pipeline.retriever.retrieve(query, mode=mode, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['chunk_id']}] score={r['score']:.4f}")
        print(f"     {r['chunk_text'][:150].strip()}...")
        print()

## 6. Run Experiments

Run all 6 automated experiments using the pre-built indexes and annotated test set.  
Results will match those in the project report.

| # | Experiment | Output |
|---|-----------|--------|
| 1 | BM25 vs Dense vs Hybrid + Factual/Semantic breakdown | `exp1_*.png` |
| 2 | Top-K sensitivity (k=5,10,20,50) | `exp2_topk_sensitivity.png` |
| 3 | Rerank Top-K sensitivity (k=1,2,3,5,10) | `exp3_rerank_topk_sensitivity.png` |
| 4 | RAG vs LLM-only baseline | `exp4_rag_vs_llm_only.png` |
| 5 | Latency profiling (per-stage breakdown) | `exp5_latency_profiling.png` |
| 6 | Reranking ablation (with vs without) | `exp6_reranking_ablation.png` |

In [ ]:
# Run all experiments (this may take a few minutes)
!python src/experiments.py

## 7. Display Experiment Results

Show all generated charts inline.

In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown

results_dir = Path("results")
chart_files = sorted(results_dir.glob("*.png"))

if not chart_files:
    print("No charts found in results/. Make sure experiments ran successfully.")
else:
    print(f"Found {len(chart_files)} charts:\n")
    for chart in chart_files:
        display(Markdown(f"### {chart.stem}"))
        display(Image(filename=str(chart), width=700))
        print()

## 8. Try Your Own Questions

Modify the question below and re-run the cell to query the system.

In [ ]:
# Change this question to anything you like
my_question = "What are the limitations of current language models?"

result = pipeline.query(my_question)

print(f"Q: {result['question']}")
print(f"\nA: {result['answer']}")
print(f"\nSources:")
for i, src in enumerate(result["sources"], 1):
    score = src.get("rerank_score", src.get("score", 0))
    print(f"  {i}. {src.get('source_paper', 'Unknown')}, p.{src.get('page_num', '?')} [score={score:.3f}]")

---

## Summary

This demo showed:
- **Pre-built corpus**: 20 ArXiv papers, pre-parsed into chunks with BM25 + FAISS indexes
- **RAG pipeline**: Hybrid retrieval → Cross-Encoder reranking → Groq LLM generation
- **Experiments**: Retrieval mode comparison, top-K sensitivity, reranking ablation, latency profiling
- **Key finding**: Hybrid retrieval (BM25 + Dense + RRF) is most robust across both factual and semantic queries

### Known Limitations
- Developed and tested on macOS (Apple Silicon). Cross-Encoder reranking falls back to bi-encoder on MPS.
- Groq free-tier API has rate limits (30 req/min, 100K tokens/day). Some experiments may hit limits.
- numpy < 2.0.0 required for faiss-cpu compatibility.